In [3]:
# Install the libraries: Gemini integration, FAISS vector store, and the text splitter
%pip install langchain-google-genai langchain-community langchain-text-splitters faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Import tools to read the key securely and to set environment variables
import os, getpass

# Ask for the Gemini API key (a box appears at the top of VS Code) instead of writing it in the code
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")

In [6]:
# Small natural-language dataset about Green IT (this is our knowledge base)
document = """
Green IT refers to the practice of designing, using, and disposing of computing resources in an environmentally responsible way. Its goal is to reduce the carbon footprint of digital technology across the whole lifecycle of hardware and software.

Data centers consume around 1 to 2 percent of global electricity. Cooling is one of their biggest energy costs, so operators use techniques such as free-air cooling, liquid cooling, and locating facilities in cold climates to reduce consumption.

Training large AI models is energy intensive. Training a single large language model can emit hundreds of tonnes of CO2. Techniques such as model distillation, quantization, and pruning reduce the compute needed and therefore the emissions.

Most of the carbon footprint of a laptop or smartphone comes from manufacturing, not from usage. Extending the lifespan of devices, repairing them, and recycling them properly is one of the most effective ways to lower the impact of IT.

Eco-design of software means writing efficient code, avoiding unnecessary computation, and reducing data transfers. Lightweight applications run on older hardware, which also extends device lifespan.
"""
print(len(document), "characters loaded")

1176 characters loaded


In [7]:
# Import the splitter that cuts text into overlapping pieces
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size = max characters per chunk; chunk_overlap = characters shared between neighbors to keep context
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# Split the document into a list of chunks
chunks = splitter.split_text(document)

# Show how many chunks we got and preview the first one
print(len(chunks), "chunks")
print(chunks[0])

3 chunks
Green IT refers to the practice of designing, using, and disposing of computing resources in an environmentally responsible way. Its goal is to reduce the carbon footprint of digital technology across the whole lifecycle of hardware and software.

Data centers consume around 1 to 2 percent of global electricity. Cooling is one of their biggest energy costs, so operators use techniques such as free-air cooling, liquid cooling, and locating facilities in cold climates to reduce consumption.


In [8]:
# Import the Gemini embedding model and the FAISS vector store
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# Embedding model: turns each chunk of text into a vector of numbers that captures its meaning
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Embed every chunk and store the vectors in FAISS (our vector database, kept in memory)
vectorstore = FAISS.from_texts(chunks, embeddings)

print("Stored", len(chunks), "chunks in FAISS")

C:\Users\abdel\AppData\Local\Temp\ipykernel_11252\3112924879.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Stored 3 chunks in FAISS


In [9]:
# Similarity search: embed the question, then return the 3 closest chunks
results = vectorstore.similarity_search("How much electricity do data centers use?", k=3)

# Print each retrieved chunk to check that the top result is relevant
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(doc.page_content)

--- Result 1 ---
Green IT refers to the practice of designing, using, and disposing of computing resources in an environmentally responsible way. Its goal is to reduce the carbon footprint of digital technology across the whole lifecycle of hardware and software.

Data centers consume around 1 to 2 percent of global electricity. Cooling is one of their biggest energy costs, so operators use techniques such as free-air cooling, liquid cooling, and locating facilities in cold climates to reduce consumption.
--- Result 2 ---
Training large AI models is energy intensive. Training a single large language model can emit hundreds of tonnes of CO2. Techniques such as model distillation, quantization, and pruning reduce the compute needed and therefore the emissions.

Most of the carbon footprint of a laptop or smartphone comes from manufacturing, not from usage. Extending the lifespan of devices, repairing them, and recycling them properly is one of the most effective ways to lower the impact 

In [12]:
# Imports for the LLM, the tool decorator, and message types
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

# Tool: the LLM can call this function to search our knowledge base
@tool
def search_documents(query: str) -> str:
    """Search the Green IT knowledge base. Use for any question about Green IT, data centers, AI energy use, devices or eco-design."""
    # Retrieve the 3 most similar chunks and join them into one text block
    docs = vectorstore.similarity_search(query, k=3)
    return "\n\n".join(d.page_content for d in docs)

# Gemini model; temperature=0 makes answers more deterministic
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

# Give the model access to the tool: it now decides by itself whether to call it
llm_with_tools = llm.bind_tools([search_documents])

# System prompt: tells the agent when to retrieve and to stay grounded in the context
system_prompt = ("You are a helpful assistant. For questions about Green IT, call the search_documents tool "
                 "and answer ONLY from the retrieved context. If the context does not contain the answer, say so. "
                 "For greetings or general chit-chat, answer directly without the tool.")

# Helper: Gemini may return text as a string or as a list of parts; this returns plain text
def to_text(content):
    if isinstance(content, str):
        return content
    return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)

# The agent: one function that handles a question from start to finish
def rag_agent(question):
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=question)]
    # Step 1: the LLM decides whether it needs the tool
    response = llm_with_tools.invoke(messages)
    if response.tool_calls:
        # Step 2: it asked for retrieval, so run the tool and give the result back
        messages.append(response)
        for call in response.tool_calls:
            result = search_documents.invoke(call["args"])
            messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
        # Step 3: the LLM writes the final answer using the retrieved context
        final = llm_with_tools.invoke(messages)
        return to_text(final.content), True
    # No tool call: the LLM answered directly
    return to_text(response.content), False

In [13]:
# Test queries: greeting, two answerable from the text, one not in the text
questions = [
    "Hello! How are you?",
    "How much electricity do data centers use and how do they cool down?",
    "Where does most of a smartphone's carbon footprint come from?",
    "What is the capital of Japan?",
]

# Run each query and show the answer and whether retrieval was used
for q in questions:
    answer, used_retrieval = rag_agent(q)
    print("Q:", q)
    print("Retrieval used:", used_retrieval)
    print("A:", answer)
    print("=" * 60)

Q: Hello! How are you?
Retrieval used: False
A: Hello! I'm doing well, thank you. How can I help you today?
Q: How much electricity do data centers use and how do they cool down?
Retrieval used: True
A: According to the retrieved documents:

* **Electricity usage:** Data centers consume around 1 to 2 percent of global electricity.
* **Cooling methods:** Operators use techniques such as free-air cooling, liquid cooling, and locating facilities in cold climates to cool down and reduce energy consumption.
Q: Where does most of a smartphone's carbon footprint come from?
Retrieval used: True
A: Most of a smartphone's carbon footprint comes from **manufacturing**, rather than from its usage. Because of this, extending the lifespan of devices, repairing them, and recycling them properly are among the most effective ways to lower their environmental impact.
Q: What is the capital of Japan?
Retrieval used: False
A: The capital of Japan is Tokyo.


## Interpretation of the results

**Query 1 (greeting):** Retrieval was not used. A greeting needs no knowledge, so the model answered directly, as instructed by the system prompt.

**Query 2 (data centers):** Retrieval was used. The agent called `search_documents`, and the answer (1 to 2 percent of global electricity; free-air cooling, liquid cooling, cold climates) matches the source text, so it is grounded in the retrieved context.

**Query 3 (smartphone):** Retrieval was used. The answer (most of the footprint comes from manufacturing, not usage) comes directly from the retrieved chunk.

**Query 4 (capital of Japan):** Retrieval was not used and the model answered from its own knowledge. This is correct, because the system prompt only requires retrieval for Green IT topics. It also shows a limit of the design: out-of-domain answers are not grounded or cited.

**Retrieval step:** The similarity search returned the data-center chunk first for the data-center question, so the Gemini embeddings capture meaning. With only 3 chunks and k=3, the search returns the whole corpus, so a larger dataset would be needed to test ranking quality.

**Overall:** The agent decides when to retrieve and grounds its answers in retrieved text, which is the main advantage of RAG over a plain LLM, whose answers come only from training memory.